# Ch 08. Activation and Loss Functions — Solutions

> This notebook contains rigorous solutions and reproducible checks for all five exercises.

## Problem 1

**Problem**: Derive directly that the derivative of sigmoid is $\sigma(x)(1-\sigma(x))$.

### Rigorous solution

Controlled factor: **derivative formula versus central difference**.  Reported metric: **maximum absolute derivative error**.  Interpretation and limitation: Differentiating 1/(1+e^-x) gives sigma(x)(1-sigma(x)); numerical differences validate the identity away from floating-point saturation.


In [ ]:
import torch
x=torch.linspace(-6,6,101,dtype=torch.float64); s=torch.sigmoid(x); e=1e-5; num=(torch.sigmoid(x+e)-torch.sigmoid(x-e))/(2*e); err=(num-s*(1-s)).abs().max().item(); print({'max_derivative_error':err}); assert err<1e-9


## Problem 2

**Problem**: Demonstrate the "dead ReLU" problem experimentally (large negative weights → a neuron stays at 0 forever).

### Rigorous solution

Controlled factor: **bias: strongly negative versus positive**.  Reported metric: **fraction of zero activations and gradient norm**.  Interpretation and limitation: With all preactivations negative, ReLU outputs and derivatives are zero, so gradient descent cannot revive that neuron on these data.


In [ ]:
import torch
x=torch.ones(32,4); w=torch.full((4,1),-5.,requires_grad=True); b=torch.tensor(-5.,requires_grad=True); a=torch.relu(x@w+b); a.sum().backward(); print({'zero_fraction':(a==0).float().mean().item(),'weight_grad_norm':w.grad.norm().item()}); assert w.grad.norm()==0


## Problem 3

**Problem**: Experiment with how GELU vs ReLU affects training in a deep MLP.

### Rigorous solution

Controlled factor: **activation: ReLU versus GELU**.  Reported metric: **final loss and first-layer gradient norm in a matched deep MLP**.  Interpretation and limitation: Weights, data, optimizer, and steps are matched. GELU preserves small negative signals while ReLU truncates them; results are specific to this reduced task.


In [ ]:
import torch
torch.manual_seed(80); X=torch.randn(128,12); y=(X[:,:4].sum(1)>0).long(); seed=torch.nn.Sequential(torch.nn.Linear(12,24),torch.nn.Linear(24,24),torch.nn.Linear(24,2)); state=seed.state_dict()
for name,act in [('relu',torch.relu),('gelu',torch.nn.functional.gelu)]:
 m=torch.nn.Sequential(torch.nn.Linear(12,24),torch.nn.Linear(24,24),torch.nn.Linear(24,2)); m.load_state_dict(state); o=torch.optim.Adam(m.parameters(),lr=.02)
 for _ in range(80): h=act(m[0](X)); h=act(m[1](h)); loss=torch.nn.functional.cross_entropy(m[2](h),y); o.zero_grad(); loss.backward(); o.step()
 print({'activation':name,'final_loss':loss.item(),'first_grad_norm':m[0].weight.grad.norm().item()})


## Problem 4

**Problem**: Compare the gradients of MSE and CE, and explain why CE is more suitable for classification.

### Rigorous solution

Controlled factor: **loss: softmax cross-entropy versus MSE on probabilities**.  Reported metric: **correct-class logit gradient magnitude**.  Interpretation and limitation: CE has gradient p-y and avoids the extra sigmoid/softmax Jacobian factor that attenuates MSE gradients when confidently wrong.


In [ ]:
import torch
z=torch.tensor([[-5.,5.]],requires_grad=True); y=torch.tensor([0]); ce=torch.nn.functional.cross_entropy(z,y); ce.backward(); gce=z.grad.clone(); z.grad=None; p=z.softmax(1); mse=((p-torch.tensor([[1.,0.]]))**2).mean(); mse.backward(); gmse=z.grad.clone(); print({'CE_grad':gce.tolist(),'MSE_grad':gmse.tolist(),'ratio':(gce.abs().max()/gmse.abs().max()).item()}); assert gce.abs().max()>gmse.abs().max()


## Problem 5

**Problem**: Compare MNIST MLP accuracy with label smoothing $\epsilon = 0, 0.1, 0.3$.

### Rigorous solution

Controlled factor: **label-smoothing epsilon: 0, 0.1, 0.3**.  Reported metric: **test accuracy and mean confidence**.  Interpretation and limitation: All conditions share a fixed synthetic classification split and initialization. Smoothing changes targets and calibration pressure; the reduced result is not presented as MNIST performance.


In [ ]:
import torch
torch.manual_seed(81); X=torch.randn(360,10); y=(X[:,:3].sum(1)>0).long(); tr=slice(0,300); te=slice(300,None); base=torch.nn.Linear(10,2).state_dict()
for eps in (0.,.1,.3):
 m=torch.nn.Linear(10,2); m.load_state_dict(base); o=torch.optim.SGD(m.parameters(),lr=.2)
 for _ in range(80): loss=torch.nn.functional.cross_entropy(m(X[tr]),y[tr],label_smoothing=eps); o.zero_grad(); loss.backward(); o.step()
 p=m(X[te]).softmax(1); print({'epsilon':eps,'accuracy':(p.argmax(1)==y[te]).float().mean().item(),'mean_confidence':p.max(1).values.mean().item()})
